# 코트 라인 세그멘테이션 학습 (Colab)

키포인트 모델(YOLO-pose)은 지점 위치가 몇 px씩 흔들려 클릭 없는 캘리브레이션에 충분히 정밀하지 않다.
대신 **코트 선 자체를 클래스별로 칠하는 세그멘테이션 모델**을 학습해, 그 출력(선 확률)을 기존 흰 선 ICP(`refine.py`)의
`ridge_map(frame)` 자리에 넣는다. 로컬에서는 `court_seg.py`가 이 노트북이 내보낸 모델을 읽는다.

데이터: `train/make_seg_dataset.py`가 만든 `court_seg_dataset.zip`
- `images/{train,val}/*.jpg` (1280x720), `masks/{train,val}/*.png` (클래스 번호, 255 = 무시), `classes.json`
- 학습 = 1103 경기, 검증 = 1008 경기 (**본 적 없는 경기**: 일반화가 핵심 지표)

준비 (한 번만)
1. 로컬 `calibration/train/court_seg_dataset.zip`을 Google Drive `내 드라이브/court_seg/`에 올린다.
2. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**.
3. **런타임 → 모두 실행**. 세션이 끊기면 다시 모두 실행하면 마지막 에폭(`last.pt`)부터 이어서 학습한다.
   학습이 끝난 뒤 다시 실행하면 학습은 건너뛰고 평가/내보내기만 다시 한다.

결과 (`내 드라이브/court_seg/baseline_seg/`)

| 파일 | 내용 | 로컬 위치 |
|---|---|---|
| `seg_best.ts` | TorchScript (메타데이터 내장). 로컬은 torch만 있으면 된다 | **`calibration/train/seg_best.ts`** |
| `seg_best.pt` | 검증 선 mIoU 최고 에폭의 state_dict | (선택) `calibration/train/seg_best.pt` |
| `seg_meta.json` | 구조, 인코더, 입력 크기, 정규화, 클래스 이름 | (선택) `calibration/train/seg_meta.json` |
| `metrics.json`, `log.csv` | 클래스별 IoU / 2 px 허용 정밀도·재현율, 에폭별 기록 | 참고용 |
| `overlays/*.jpg`, `aug_preview.jpg` | 검증 예측 겹쳐 그리기, 증강 확인 | 참고용 |
| `last.pt` | 이어 학습용 체크포인트 (모델+옵티마이저+스케줄러) | 복사하지 않음 |

주의: 기존 키포인트 모델이 `train/best.pt`라 이름이 겹치지 않도록 세그멘테이션 모델은 `seg_` 접두사를 붙인다.

## 모델, 손실, 증강을 이렇게 고른 이유

**U-Net + ResNet-34 (ImageNet 사전학습), `segmentation_models_pytorch==0.5.0`**
- 선은 1~3 px 폭이다 (먼 선은 1280 폭에서 약 2 px). U-Net은 1/2 해상도 스킵 연결까지 디코더로 올려 가는 구조를 그대로
  살리므로 얇은 구조에 강하다. DeepLabV3+ 계열은 1/4 해상도에서 업샘플해 2 px 선이 뭉개진다.
- ResNet-34는 약 24M 파라미터로 T4에서 1280x736 배치 8을 AMP로 무리 없이 돌린다(VRAM 약 8~10 GB).
  추론은 한 장에 약 227 GFLOP: 로컬 CPU(4 스레드)에서 실측 약 20초/프레임. 캘리브레이션은 시점 5개 정도만 보므로
  감당할 수 있고, 일반적인 최신 노트북 CPU에서는 몇 초 수준이다. 더 빨라야 하면 설정 셀의 ENCODER만 바꾸면 된다. EfficientNet-B2는 파라미터는 적지만
  depthwise 합성곱 때문에 T4 학습이 더 느리고 메모리를 더 쓴다. Unet++는 고해상도 중첩 디코더라 1280 폭에서 2~3배 무겁다.
- TorchScript로 trace가 되므로 로컬에 smp를 설치하지 않아도 된다.

**입력 1280x720 그대로** (아래·오른쪽을 32의 배수 1280x736으로 패딩, 패딩 영역 마스크는 255=무시). 축소하면 먼 선이 1 px 아래로 사라진다.

**손실 = 클래스 가중 교차 엔트로피 + Dice (선 클래스만, ignore_index=255)**
- 배경이 95% 이상이라 가중치 없는 CE는 "전부 배경"으로 수렴한다. 가중치는 학습 마스크 픽셀 빈도로 정한다:
  `w_c = clip(sqrt(median_freq / freq_c), 0.05, 10)` (제곱근으로 완화해 선이 두꺼워지는 것을 억제).
- Dice는 클래스별 겹침을 직접 최적화해 픽셀 수가 매우 적은 클래스(spots, 먼 골라인)를 살린다.

**증강 (albumentations 2.0.8)**
- 색/화질: 밝기·대비, 감마, 색상·채도, 블러/모션 블러, 노이즈, JPEG 압축, 그림자 → 다른 경기장 조명/카메라로 일반화.
- 기하: 작은 스케일(±10%)·회전(±3°)·이동, 약한 원근 → 카메라 위치 차이. 이미지와 마스크에 같이 적용 (마스크는 최근접 보간, 밖은 무시).
- **좌우 반전 없음**: `sideline_left/right` 등은 화면 배치에 묶인 클래스다. 미러 경기는 데이터셋에서 이미 정규 배치로 뒤집혀 있다.

**학습**: AdamW(lr 5e-4, wd 1e-4), 1 에폭 워밍업 + 코사인, AMP, 최대 60 에폭, 검증 선 mIoU가 12 에폭 동안 안 오르면 조기 종료.
검증 지표: 클래스별 IoU(배경 제외 평균 = 선 mIoU, 이것으로 best 선택), 2 px 허용 정밀도/재현율/F1 (ICP는 몇 px 안의 선만 쓰므로 실제 용도에 더 가깝다).

**예상 시간 (T4)**: 학습 이미지 약 2천 장 기준 에폭당 약 2~3분 → 40~60 에폭에 약 1.5~2.5시간. 첫 에폭 뒤 예상 시간을 출력한다.
무료 세션이 끊겨도 다시 모두 실행하면 이어진다.

In [ ]:
# 1. GPU 확인 (없으면 여기서 멈춘다: CPU로는 며칠 걸린다)
import torch, subprocess
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경 > T4 GPU 로 바꾼 뒤 다시 실행하세요'

In [ ]:
# 2. Drive 연결, 경로
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/court_seg'
ZIP = os.path.join(DRIVE, 'court_seg_dataset.zip')
OUT = os.path.join(DRIVE, 'baseline_seg')   # 체크포인트, 결과물 전부 여기 (세션이 끊겨도 남는다)
DATA_DIR = '/content/court_seg'             # 로컬 디스크 (Drive에서 직접 읽으면 느리다)
assert os.path.exists(ZIP), f'데이터가 없습니다: {ZIP} (로컬 calibration/train/court_seg_dataset.zip을 올리세요)'
os.makedirs(OUT, exist_ok=True)
print('데이터:', ZIP, f'{os.path.getsize(ZIP) / 1e6:.0f} MB')

In [ ]:
# 3. 패키지 (버전 고정: 로컬 court_seg.py의 대체 경로와 같은 smp 버전)
!pip install -q segmentation-models-pytorch==0.5.0 albumentations==2.0.8

In [ ]:
# 4. 설정
import torch
SEED = 0
ARCH, ENCODER, ENCODER_WEIGHTS = 'Unet', 'resnet34', 'imagenet'
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet, RGB 0~1 기준
PAD_TO = 32              # U-Net 다운샘플 5단: 입력 크기를 32의 배수로 (720 -> 736 패딩)
EPOCHS = 60
PATIENCE = 12            # 검증 선 mIoU가 이만큼 안 오르면 조기 종료
BATCH, VAL_BATCH = 8, 4  # T4 16 GB에서 1280x736 AMP 기준. 메모리 부족이면 6으로
LR, WD, WARMUP_EPOCHS = 5e-4, 1e-4, 1
DICE_W = 1.0             # 손실 = CE(가중) + DICE_W * Dice
TOL_PX = 2               # 허용 오차 지표의 반경 (px, 1280x720 기준)
NUM_WORKERS = 2          # Colab 무료는 CPU 2개
N_OVERLAY = 8            # Drive에 저장할 검증 예측 그림 수
MAX_TRAIN = MAX_VAL = None  # 빠른 점검용으로 장 수를 줄일 때만
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

In [ ]:
# 5. 데이터를 로컬 디스크에 풀고 클래스 정보를 읽는다 (클래스 수/이름은 classes.json에서: 하드코딩하지 않는다)
import os, json, zipfile, glob
if not glob.glob(os.path.join(DATA_DIR, '**', 'classes.json'), recursive=True):
    zipfile.ZipFile(ZIP).extractall(DATA_DIR)
DATA_ROOT = os.path.dirname(sorted(glob.glob(os.path.join(DATA_DIR, '**', 'classes.json'), recursive=True), key=len)[0])
CLS = json.load(open(os.path.join(DATA_ROOT, 'classes.json'), encoding='utf-8'))
NAMES = list(CLS['names'])
NC = len(NAMES)
IGNORE = int(CLS.get('ignore_index', 255))
IMG_W, IMG_H = CLS.get('image_size', [1280, 720])
LINE_IDS = [i for i, n in enumerate(NAMES) if n != 'background']
print('클래스', NC, NAMES, '/ 무시', IGNORE, '/ 크기', IMG_W, 'x', IMG_H)
for split in ('train', 'val'):
    print(split, len(glob.glob(f'{DATA_ROOT}/images/{split}/*.jpg')), '장 / 마스크', len(glob.glob(f'{DATA_ROOT}/masks/{split}/*.png')))

In [ ]:
# 6. 데이터셋, 증강, GPU 전처리
import cv2, numpy as np, albumentations as A
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
cv2.setNumThreads(0)

def stems(split):
    return sorted(os.path.splitext(f)[0] for f in os.listdir(f'{DATA_ROOT}/images/{split}') if f.endswith('.jpg'))

# 기하 증강 밖으로 나간 영역: 이미지는 검정, 마스크는 무시(255). 좌우 반전은 넣지 않는다 (left/right 클래스가 배치에 묶여 있다)
train_tf = A.Compose([
    A.Affine(scale=(0.9, 1.1), rotate=(-3, 3), translate_percent={'x': (-0.04, 0.04), 'y': (-0.04, 0.04)},
             interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST,
             border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=IGNORE, p=0.7),
    A.Perspective(scale=(0.01, 0.03), keep_size=True, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST,
                  border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=IGNORE, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.8),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=25, val_shift_limit=15, p=0.5),
    A.RandomShadow(shadow_roi=(0, 0.3, 1, 1), num_shadows_limit=(1, 2), shadow_dimension=5, p=0.2),
    A.OneOf([A.GaussianBlur(blur_limit=(3, 5), sigma_limit=(0.3, 1.0)), A.MotionBlur(blur_limit=(3, 7))], p=0.25),
    A.GaussNoise(std_range=(0.01, 0.04), p=0.3),
    A.ImageCompression(quality_range=(40, 90), p=0.4),
])


class SegDS(Dataset):
    def __init__(self, split, tf=None, limit=None):
        self.split, self.tf = split, tf
        self.stems = stems(split)[:limit]

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, i):
        s = self.stems[i]
        img = cv2.cvtColor(cv2.imread(f'{DATA_ROOT}/images/{self.split}/{s}.jpg', cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(f'{DATA_ROOT}/masks/{self.split}/{s}.png', cv2.IMREAD_UNCHANGED)
        if img.shape[:2] != (IMG_H, IMG_W):  # 안전장치: 데이터셋은 이미 이 크기
            img = cv2.resize(img, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
        if mask.shape[:2] != (IMG_H, IMG_W):
            mask = cv2.resize(mask, (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST)
        if self.tf is not None:
            r = self.tf(image=img, mask=mask)
            img, mask = r['image'], r['mask']
        # uint8 그대로 넘기고 정규화는 GPU에서 (CPU 2개라 데이터 로딩이 병목이 되기 쉽다)
        return torch.from_numpy(np.ascontiguousarray(img)), torch.from_numpy(np.ascontiguousarray(mask))


def to_input(img, mask=None, device=DEVICE):
    # (N,H,W,3) uint8 RGB -> 정규화한 (N,3,H',W') float, 아래/오른쪽을 PAD_TO 배수로 0 패딩 (court_seg.py와 같은 전처리)
    x = img.to(device, non_blocking=True).permute(0, 3, 1, 2).float()
    mean = torch.tensor(MEAN, device=device).view(1, 3, 1, 1) * 255
    std = torch.tensor(STD, device=device).view(1, 3, 1, 1) * 255
    x = (x - mean) / std
    ph, pw = (-x.shape[2]) % PAD_TO, (-x.shape[3]) % PAD_TO
    x = F.pad(x, (0, pw, 0, ph), value=0.0)
    if mask is None:
        return x
    y = F.pad(mask.to(device, non_blocking=True).long(), (0, pw, 0, ph), value=IGNORE)
    return x, y


PALETTE = np.zeros((256, 3), np.uint8)  # 클래스 색 (BGR), 0 = 배경 검정, 255 = 무시 회색
_h = np.zeros((NC, 1, 3), np.uint8)
_h[:, 0, 0] = (np.arange(NC) * 180 // max(NC - 1, 1)) % 180
_h[:, 0, 1:] = 255
PALETTE[:NC] = cv2.cvtColor(_h, cv2.COLOR_HSV2BGR)[:, 0]
PALETTE[0] = 0
PALETTE[IGNORE] = 90


def overlay(img_rgb, labels):
    # 선 픽셀만 클래스 색으로 칠한다 (그대로 두면 2 px 선이 안 보여 3x3으로 두껍게)
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR) // 2
    lab = labels.astype(np.uint8)
    thick = cv2.dilate(np.where((lab > 0) & (lab != IGNORE), lab, 0).astype(np.uint8), np.ones((3, 3), np.uint8))
    m = thick > 0
    bgr[m] = PALETTE[thick[m]]
    bgr[lab == IGNORE] = bgr[lab == IGNORE] // 2 + 40
    return bgr


train_ds = SegDS('train', train_tf, MAX_TRAIN)
val_ds = SegDS('val', None, MAX_VAL)
pin = DEVICE.type == 'cuda'
train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin, drop_last=True,
                      persistent_workers=NUM_WORKERS > 0)
val_dl = DataLoader(val_ds, VAL_BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
print('학습', len(train_ds), '장 /', len(train_dl), '배치,  검증', len(val_ds), '장')

In [ ]:
# 7. 증강 확인: 같은 학습 이미지 4개를 증강해 마스크와 겹쳐 Drive에 저장한다 (선이 이미지와 같이 움직이는지 본다)
rng = np.random.default_rng(SEED)
tiles = []
for i in rng.choice(len(train_ds), size=min(4, len(train_ds)), replace=False):
    img, mask = train_ds[int(i)]
    tiles.append(cv2.resize(overlay(img.numpy(), mask.numpy()), (IMG_W // 2, IMG_H // 2), interpolation=cv2.INTER_AREA))
while len(tiles) % 2:
    tiles.append(np.zeros_like(tiles[0]))
grid = np.vstack([np.hstack(tiles[k:k + 2]) for k in range(0, len(tiles), 2)])
cv2.imwrite(os.path.join(OUT, 'aug_preview.jpg'), grid)
try:
    from IPython.display import Image, display
    display(Image(os.path.join(OUT, 'aug_preview.jpg'), width=900))
except ImportError:
    print('저장:', os.path.join(OUT, 'aug_preview.jpg'))

In [ ]:
# 8. 클래스 가중치: 학습 마스크의 픽셀 빈도로 정한다 (한 번 계산해 Drive에 두고, 이어 학습할 때 같은 값을 쓴다)
CW_PATH = os.path.join(OUT, 'class_weights.json')
if os.path.exists(CW_PATH) and json.load(open(CW_PATH))['names'] == NAMES:
    cw = json.load(open(CW_PATH))
else:
    counts = np.zeros(256, np.int64)
    for s in stems('train'):
        counts += np.bincount(cv2.imread(f'{DATA_ROOT}/masks/train/{s}.png', cv2.IMREAD_UNCHANGED).ravel(), minlength=256)
    unknown = int(counts[NC:].sum() - counts[IGNORE])
    assert unknown == 0, f'마스크에 classes.json에 없는 값이 있습니다 ({unknown} px)'
    c = counts[:NC].astype(np.float64)
    freq = c / c.sum()
    med = np.median(freq[freq > 0])
    w = np.where(freq > 0, np.clip(np.sqrt(med / np.maximum(freq, 1e-12)), 0.05, 10.0), 1.0)
    cw = {'names': NAMES, 'pixels': c.astype(int).tolist(), 'ignored_pixels': int(counts[IGNORE]),
          'freq': freq.tolist(), 'weights': w.tolist(), 'rule': 'clip(sqrt(median_freq / freq), 0.05, 10)'}
    json.dump(cw, open(CW_PATH, 'w'), indent=1)
for n, f, w in zip(NAMES, cw['freq'], cw['weights']):
    print(f'{n:20s} 빈도 {f * 100:8.4f}%  가중치 {w:6.2f}')

In [ ]:
# 9. 모델, 손실, 옵티마이저, 스케줄러. Drive에 last.pt가 있으면 거기서 이어간다
import math, random, time, csv
import torch.nn as nn
import segmentation_models_pytorch as smp
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

def build_model(weights=ENCODER_WEIGHTS):
    return getattr(smp, ARCH)(encoder_name=ENCODER, encoder_weights=weights, in_channels=3, classes=NC)

model = build_model().to(DEVICE)
ce = nn.CrossEntropyLoss(weight=torch.tensor(cw['weights'], dtype=torch.float32, device=DEVICE), ignore_index=IGNORE)
dice = smp.losses.DiceLoss('multiclass', classes=LINE_IDS, from_logits=True, ignore_index=IGNORE)

def loss_fn(logits, y):
    logits = logits.float()  # 손실은 FP32로 (AMP에서 작은 클래스 확률이 0으로 떨어지지 않게)
    return ce(logits, y) + DICE_W * dice(logits, y)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
TOTAL = EPOCHS * len(train_dl)
WARM = max(1, WARMUP_EPOCHS * len(train_dl))

def lr_lambda(it):  # 반복마다: 선형 워밍업 뒤 코사인으로 LR의 1%까지
    if it < WARM:
        return (it + 1) / WARM
    p = min(1.0, (it - WARM) / max(1, TOTAL - WARM))
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * p))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
LAST, BEST = os.path.join(OUT, 'last.pt'), os.path.join(OUT, 'seg_best.pt')
state = {'epoch': 0, 'best': -1.0, 'best_epoch': 0, 'bad': 0, 'done': False, 'history': []}
if os.path.exists(LAST):
    ck = torch.load(LAST, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
    state = ck['state']
    print(f"이어서 학습: {state['epoch']} 에폭까지 완료, best 선 mIoU {state['best']:.4f} (에폭 {state['best_epoch']})"
          + (' / 이미 끝남 -> 평가만' if state['done'] else ''))
else:
    print('처음부터 학습')
print(f'{ARCH}/{ENCODER}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M 파라미터')

In [ ]:
# 10. 검증: 클래스별 IoU (혼동 행렬) + TOL_PX 허용 정밀도/재현율
@torch.no_grad()
def evaluate(model, dl, tol=TOL_PX):
    model.eval()
    conf = torch.zeros(NC * NC, dtype=torch.float64, device=DEVICE)
    s = {k: torch.zeros(NC, dtype=torch.float64, device=DEVICE) for k in ('tp_p', 'n_p', 'tp_g', 'n_g')}
    a = torch.zeros(4, dtype=torch.float64, device=DEVICE)  # 클래스 구분 없이 '선' 하나로 본 허용 지표
    k = 2 * tol + 1
    for img, mask in dl:
        x, y = to_input(img, mask)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            pred = model(x).argmax(1)
        valid = y != IGNORE
        conf += torch.bincount((y[valid] * NC + pred[valid]), minlength=NC * NC).double()
        v = valid.unsqueeze(1).float()
        P = F.one_hot(pred, NC).permute(0, 3, 1, 2).float() * v
        G = F.one_hot(torch.where(valid, y, 0), NC).permute(0, 3, 1, 2).float() * v
        Pd, Gd = F.max_pool2d(P, k, 1, tol), F.max_pool2d(G, k, 1, tol)  # tol px 팽창
        s['tp_p'] += (P * Gd).sum((0, 2, 3)); s['n_p'] += P.sum((0, 2, 3))
        s['tp_g'] += (G * Pd).sum((0, 2, 3)); s['n_g'] += G.sum((0, 2, 3))
        Pa, Ga = P[:, LINE_IDS].amax(1, keepdim=True), G[:, LINE_IDS].amax(1, keepdim=True)
        Pad, Gad = F.max_pool2d(Pa, k, 1, tol), F.max_pool2d(Ga, k, 1, tol)
        a += torch.stack([(Pa * Gad).sum(), Pa.sum(), (Ga * Pad).sum(), Ga.sum()]).double()
    cm = conf.view(NC, NC).cpu().numpy()  # 행 = 정답, 열 = 예측
    inter = np.diag(cm); gt = cm.sum(1); pr = cm.sum(0)
    with np.errstate(invalid='ignore', divide='ignore'):
        iou = np.where(gt > 0, inter / (gt + pr - inter), np.nan)  # 검증에 없는 클래스는 평균에서 뺀다
        sn = {k: v.cpu().numpy() for k, v in s.items()}
        prec = np.where(sn['n_p'] > 0, sn['tp_p'] / sn['n_p'], np.nan)
        rec = np.where(sn['n_g'] > 0, sn['tp_g'] / sn['n_g'], np.nan)
        f1 = 2 * prec * rec / (prec + rec)
    an = a.cpu().numpy()
    ap, ar = an[0] / max(an[1], 1), an[2] / max(an[3], 1)
    return {'line_miou': float(np.nanmean(iou[LINE_IDS])), 'line_tol_f1': float(np.nanmean(f1[LINE_IDS])),
            'any_line_tol': {'precision': float(ap), 'recall': float(ar), 'f1': float(2 * ap * ar / max(ap + ar, 1e-9))},
            'iou': iou, 'tol_precision': prec, 'tol_recall': rec, 'tol_f1': f1, 'gt_pixels': gt, 'pred_pixels': pr}

def class_table(ev):
    print(f"{'클래스':20s} {'IoU':>7s} {f'정밀({TOL_PX}px)':>10s} {f'재현({TOL_PX}px)':>10s} {'F1':>7s} {'정답 px':>11s} {'예측 px':>11s}")
    for i, n in enumerate(NAMES):
        print(f"{n:20s} {ev['iou'][i]:7.4f} {ev['tol_precision'][i]:10.4f} {ev['tol_recall'][i]:10.4f} {ev['tol_f1'][i]:7.4f}"
              f" {int(ev['gt_pixels'][i]):11d} {int(ev['pred_pixels'][i]):11d}")
    t = ev['any_line_tol']
    print(f"선 mIoU(배경 제외) {ev['line_miou']:.4f} / 선 허용 F1 평균 {ev['line_tol_f1']:.4f} / "
          f"클래스 무시 선: 정밀 {t['precision']:.4f} 재현 {t['recall']:.4f} F1 {t['f1']:.4f}")

In [ ]:
# 11. 학습. 매 에폭 last.pt (모델+옵티마이저+스케줄러+상태)를 Drive에 덮어쓴다 -> 끊겨도 다시 모두 실행하면 이어진다
def atomic_save(obj, path):
    torch.save(obj, path + '.tmp')
    os.replace(path + '.tmp', path)

def write_log():
    if not state['history']:
        return
    keys = list(state['history'][-1].keys())
    with open(os.path.join(OUT, 'log.csv'), 'w', newline='') as f:
        wr = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
        wr.writeheader(); wr.writerows(state['history'])

session_start = time.time()
first_epoch = state['epoch']
while not state['done'] and state['epoch'] < EPOCHS:
    t0 = time.time()
    model.train()
    tl, n = 0.0, 0
    for img, mask in train_dl:
        x, y = to_input(img, mask)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
        loss = loss_fn(logits, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item(); n += 1
    ev = evaluate(model, val_dl)
    state['epoch'] += 1
    ep = state['epoch']
    row = {'epoch': ep, 'train_loss': round(tl / max(n, 1), 5), 'lr': opt.param_groups[0]['lr'],
           'val_line_miou': round(ev['line_miou'], 5), 'val_line_tol_f1': round(ev['line_tol_f1'], 5),
           'val_any_line_tol_f1': round(ev['any_line_tol']['f1'], 5), 'sec': round(time.time() - t0, 1)}
    row.update({f'iou_{nm}': round(float(ev['iou'][i]), 5) for i, nm in enumerate(NAMES)})
    state['history'].append(row)
    if ev['line_miou'] > state['best']:
        state['best'], state['best_epoch'], state['bad'] = ev['line_miou'], ep, 0
        atomic_save(model.state_dict(), BEST)  # 순수 state_dict
        mark = ' *best'
    else:
        state['bad'] += 1
        mark = ''
    if state['bad'] >= PATIENCE or ep >= EPOCHS:
        state['done'] = True
    atomic_save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                 'scaler': scaler.state_dict(), 'state': state}, LAST)
    write_log()
    print(f"[{ep:3d}/{EPOCHS}] loss {row['train_loss']:.4f}  선 mIoU {ev['line_miou']:.4f}  허용 F1 {ev['line_tol_f1']:.4f}"
          f"  {row['sec']:.0f}s{mark}" + ('  (조기 종료)' if state['bad'] >= PATIENCE else ''))
    if ep == first_epoch + 1:
        print(f"  예상: 남은 {EPOCHS - ep} 에폭 x {row['sec']:.0f}s = 최대 {(EPOCHS - ep) * row['sec'] / 3600:.1f}시간 (조기 종료하면 더 짧다)")
print(f"학습 끝: best 선 mIoU {state['best']:.4f} (에폭 {state['best_epoch']}), 이번 세션 {(time.time() - session_start) / 60:.0f}분")

In [ ]:
# 12. best 모델로 최종 평가: 클래스별 표, metrics.json, 검증 예측 그림
import datetime, albumentations
model.load_state_dict(torch.load(BEST, map_location=DEVICE, weights_only=True))
ev = evaluate(model, val_dl)
class_table(ev)

nan2none = lambda v: None if v is None or (isinstance(v, float) and math.isnan(v)) else round(float(v), 5)
metrics = {
    'time': datetime.datetime.now().isoformat(timespec='seconds'),
    'torch': torch.__version__, 'smp': smp.__version__, 'albumentations': albumentations.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'model': {'arch': ARCH, 'encoder': ENCODER, 'encoder_weights': ENCODER_WEIGHTS,
              'input': [IMG_W, IMG_H], 'pad_to': PAD_TO},
    'train': {'epochs_run': state['epoch'], 'best_epoch': state['best_epoch'], 'max_epochs': EPOCHS, 'patience': PATIENCE,
              'batch': BATCH, 'lr': LR, 'wd': WD, 'dice_w': DICE_W, 'loss': 'weighted CE + Dice(line classes)',
              'class_weights': cw['weights'], 'n_train': len(train_ds), 'n_val': len(val_ds)},
    'data': {k: v for k, v in CLS.items() if k != 'names'},
    'val': {'line_miou': nan2none(ev['line_miou']), 'line_tol_f1': nan2none(ev['line_tol_f1']), 'tol_px': TOL_PX,
            'any_line_tol': ev['any_line_tol'],
            'per_class': {n: {'iou': nan2none(ev['iou'][i]), 'tol_precision': nan2none(ev['tol_precision'][i]),
                              'tol_recall': nan2none(ev['tol_recall'][i]), 'tol_f1': nan2none(ev['tol_f1'][i]),
                              'gt_pixels': int(ev['gt_pixels'][i]), 'pred_pixels': int(ev['pred_pixels'][i])}
                          for i, n in enumerate(NAMES)}},
}
json.dump(metrics, open(os.path.join(OUT, 'metrics.json'), 'w', encoding='utf-8'), indent=1, ensure_ascii=False)
print('저장:', os.path.join(OUT, 'metrics.json'))

# 검증 예측 그림: 왼쪽 정답, 오른쪽 예측 (반 크기)
ov_dir = os.path.join(OUT, 'overlays')
os.makedirs(ov_dir, exist_ok=True)
model.eval()
for i in np.unique(np.linspace(0, len(val_ds) - 1, N_OVERLAY).astype(int)):
    img, mask = val_ds[int(i)]
    with torch.no_grad(), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        pred = model(to_input(img[None]))[0].argmax(0)[:IMG_H, :IMG_W].cpu().numpy()
    im = img.numpy()
    pair = np.hstack([overlay(im, mask.numpy()), overlay(im, pred)])
    cv2.imwrite(os.path.join(ov_dir, f'{val_ds.stems[int(i)]}.jpg'),
                cv2.resize(pair, (IMG_W, IMG_H // 2), interpolation=cv2.INTER_AREA), [cv2.IMWRITE_JPEG_QUALITY, 90])
print('저장:', ov_dir)

In [ ]:
# 13. 내보내기: seg_meta.json + TorchScript(seg_best.ts, 메타데이터 내장). 로컬 court_seg.py는 smp 없이 .ts만으로 돈다
META_PATH, TS_PATH = os.path.join(OUT, 'seg_meta.json'), os.path.join(OUT, 'seg_best.ts')
H_PAD, W_PAD = IMG_H + (-IMG_H) % PAD_TO, IMG_W + (-IMG_W) % PAD_TO
meta = {
    'arch': ARCH, 'encoder': ENCODER, 'smp_version': smp.__version__, 'torch_version': torch.__version__,
    'input_size': [IMG_W, IMG_H], 'pad_to': PAD_TO, 'padded_size': [W_PAD, H_PAD],
    'color': 'RGB', 'mean': list(MEAN), 'std': list(STD),
    'preprocess': 'resize to input_size, RGB, (x/255 - mean)/std, zero-pad bottom/right to pad_to multiple',
    'output': 'logits (N, C, H_pad, W_pad); softmax over C, crop [:H, :W]',
    'names': NAMES, 'line_classes': LINE_IDS, 'ignore_index': IGNORE,
    'layout': 'canonical (1008). calibration mirror:true camera -> flip frame horizontally before, flip output back',
    'best_epoch': state['best_epoch'], 'val_line_miou': nan2none(ev['line_miou']),
}
json.dump(meta, open(META_PATH, 'w', encoding='utf-8'), indent=1, ensure_ascii=False)

cpu_model = build_model(weights=None)
cpu_model.load_state_dict(torch.load(BEST, map_location='cpu', weights_only=True))
cpu_model.eval()
example = torch.zeros(1, 3, H_PAD, W_PAD)
with torch.no_grad():
    ts = torch.jit.trace(cpu_model, example)
torch.jit.save(ts, TS_PATH, _extra_files={'meta.json': json.dumps(meta, ensure_ascii=False)})

# 검증: 다시 읽은 TorchScript와 원래 모델의 출력이 같은지 실제 검증 이미지로 확인
extra = {'meta.json': ''}
ts2 = torch.jit.load(TS_PATH, map_location='cpu', _extra_files=extra)
assert json.loads(extra['meta.json'])['names'] == NAMES
x = to_input(val_ds[0][0][None], device='cpu')
with torch.no_grad():
    d = (ts2(x) - cpu_model(x)).abs().max().item()
print(f'TorchScript 최대 차이 {d:.2e}')
assert d < 1e-3, 'TorchScript 출력이 원래 모델과 다릅니다'
for p in (TS_PATH, BEST, META_PATH, os.path.join(OUT, 'metrics.json')):
    print(f'{p}  {os.path.getsize(p) / 1e6:.1f} MB')
print('\n로컬로 복사: seg_best.ts -> D:\\02_Projects\\futsal\\calibration\\train\\seg_best.ts')
print('(선택, smp 설치 시) seg_best.pt, seg_meta.json -> calibration\\train\\')

In [ ]:
# 14. (선택) 바로 내려받기 - Drive에서 받아도 된다
from google.colab import files
for f in ('seg_best.ts', 'seg_meta.json', 'metrics.json'):
    files.download(os.path.join(OUT, f))